# 01 — Data Acquisition & Exploratory Analysis

Ingest aggregated natural gas consumption by postal code, regional HDD data, and MPAC property tax roll summaries. Filter to residential consumers and perform exploratory analysis on gas-temperature relationships.

## 1.1 — Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Environment ready.')

## 1.2 — Generate Synthetic Gas Consumption Data

Simulate 150 postal codes × 24 months of residential natural gas consumption.
Each postal code has a true thermal slope, baseload, and customer count.

In [ ]:
np.random.seed(42)

N_POSTAL = 150
N_MONTHS = 24
MONTHS = pd.date_range('2024-01-01', periods=N_MONTHS, freq='MS')

# Monthly HDD for southern Ontario (approximate)
hdd_template = np.array([650, 580, 480, 250, 80, 10, 0, 0, 30, 200, 400, 600])
hdd_monthly = np.tile(hdd_template, 2) + np.random.normal(0, 25, N_MONTHS)
hdd_monthly = np.maximum(hdd_monthly, 0)

# Postal code characteristics
postal_codes = [f'N2{chr(65+i//26)}{chr(65+i%26):1s}{np.random.randint(1,9)}{chr(65+np.random.randint(0,26))}{np.random.randint(0,9)}'
                for i in range(N_POSTAL)]

# True thermal slopes (GJ per HDD) — range from well-insulated to leaky
true_slopes = np.random.uniform(0.02, 0.12, N_POSTAL)
# Customer counts per postal code
customer_counts = np.random.randint(10, 40, N_POSTAL)
# Baseload (GJ/month — DHW, cooking)
baseloads = np.random.uniform(1.5, 4.0, N_POSTAL) * customer_counts

print(f'Generated {N_POSTAL} postal codes with {N_MONTHS} months of data.')
print(f'HDD range: {hdd_monthly.min():.0f} to {hdd_monthly.max():.0f}')

## 1.3 — Assemble Gas Consumption DataFrame

In [ ]:
rows = []
for j, pc in enumerate(postal_codes):
    for i, month in enumerate(MONTHS):
        gas = (true_slopes[j] * customer_counts[j] * hdd_monthly[i]
               + baseloads[j]
               + np.random.normal(0, baseloads[j] * 0.15))
        gas = max(gas, baseloads[j] * 0.3)  # floor at minimal baseload
        rows.append({
            'postal_code': pc,
            'month': month,
            'gas_gj': round(gas, 2),
            'hdd': round(hdd_monthly[i], 1),
            'customer_count': customer_counts[j],
            'consumer_type': 'residential'
        })

gas_df = pd.DataFrame(rows)
print(f'Gas consumption DataFrame: {gas_df.shape}')
gas_df.head(10)

## 1.4 — Generate MPAC Property Tax Roll Summary

Synthetic building stock characteristics per postal code: footprint, storeys, structure type, basement indicator.

In [ ]:
structure_types = ['detached', 'semi-detached', 'row', 'low-rise-apt']
type_weights = [0.50, 0.20, 0.25, 0.05]

mpac_rows = []
for j, pc in enumerate(postal_codes):
    stype = np.random.choice(structure_types, p=type_weights)
    storeys = np.random.choice([1, 1.5, 2, 2.5, 3],
                                p=[0.25, 0.15, 0.35, 0.15, 0.10])
    footprint = {'detached': np.random.uniform(80, 160),
                 'semi-detached': np.random.uniform(60, 120),
                 'row': np.random.uniform(45, 90),
                 'low-rise-apt': np.random.uniform(200, 500)}[stype]
    has_basement = np.random.choice([1.0, 0.5, 0.0], p=[0.55, 0.30, 0.15])

    mpac_rows.append({
        'postal_code': pc,
        'structure_type': stype,
        'avg_storeys': storeys,
        'avg_footprint_m2': round(footprint, 1),
        'basement_fraction': has_basement,
        'property_count': customer_counts[j],
        'true_slope': true_slopes[j],  # ground truth for validation
    })

mpac_df = pd.DataFrame(mpac_rows)
print(f'MPAC summary: {mpac_df.shape}')
mpac_df.head(10)

## 1.5 — Exploratory Analysis: Gas vs. HDD

In [ ]:
# Pick 6 representative postal codes spanning the slope range
sorted_idx = np.argsort(true_slopes)
sample_idx = sorted_idx[np.linspace(0, N_POSTAL-1, 6, dtype=int)]
sample_pcs = [postal_codes[i] for i in sample_idx]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=False)
for ax, pc in zip(axes.flat, sample_pcs):
    subset = gas_df[gas_df.postal_code == pc]
    ax.scatter(subset.hdd, subset.gas_gj, alpha=0.7, s=30)
    ax.set_title(pc, fontsize=10)
    ax.set_xlabel('Monthly HDD')
    ax.set_ylabel('Gas (GJ)')

plt.suptitle('Gas Consumption vs. HDD — Sample Postal Codes', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 1.6 — Distribution of Customer Counts and Building Types

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(customer_counts, bins=20, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Residential Customers per Postal Code')
ax1.set_ylabel('Count')
ax1.set_title('Customer Count Distribution')

mpac_df.structure_type.value_counts().plot.barh(ax=ax2, color='steelblue')
ax2.set_xlabel('Number of Postal Codes')
ax2.set_title('Dominant Structure Type')

plt.tight_layout()
plt.show()

## 1.7 — Monthly HDD Pattern

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(N_MONTHS), hdd_monthly, color='coral', alpha=0.8)
ax.set_xticks(range(N_MONTHS))
ax.set_xticklabels([m.strftime('%b %y') for m in MONTHS], rotation=45, ha='right')
ax.set_ylabel('HDD18')
ax.set_title('Monthly Heating Degree Days - Regional Weather Station')
plt.tight_layout()
plt.show()

## 1.8 — Save Intermediate Data

In [ ]:
gas_df.to_csv('data/ces_gas_consumption.csv', index=False)
mpac_df.to_csv('data/ces_mpac_summary.csv', index=False)
print('Saved: ces_gas_consumption.csv, ces_mpac_summary.csv')

---
**Next:** Notebook 02 runs the HDD regression per postal code, normalizes by building stock, and produces the thermal intensity metric.